# 07 — Scale comparison — SKIPPED_PREREQUISITE

This is an executable gate record, not a Stage-3 science run. Stage 2 required
the replication-failure path, so no model is loaded and no cross-scale P1 measurement
is attempted. The cell below fails closed unless the persisted hard-gate
decision explicitly prohibits Stage 3.

In [1]:
import json
import os
from pathlib import Path

ROOT = Path('/home/jovyan/j-space-thoughts')
os.chdir(ROOT)

metrics = json.loads((ROOT / 'results/metrics.json').read_text())
repair = metrics['repair_v2']
stage2 = repair['stage2_recalibration']
ledger = repair['gate_ledger']
assert stage2['stage4_required'] is True
assert stage2['stage3_allowed'] is False
assert ledger['stage3_science'] == 'SKIPPED_PREREQUISITE'

criteria = stage2.get('criteria')
if not isinstance(criteria, dict):
    criteria = {
        'g_swap_reverified': stage2['g_swap_reverification']['status'] == 'PASS',
        'controls_fire': stage2['controls_fire']['status'] == 'PASS',
        'matched_random_specificity': stage2['random_pair_null']['status'] == 'PASS',
        'absent_coordinate_specificity': stage2['absent_coordinate_null']['status'] == 'PASS',
        'capability_preserved': stage2['capability']['status'] == 'PASS',
        'g_pos_reproduced': stage2['g_pos']['status'] == 'PASS',
    }

blocking_criteria = {
    name: value
    for name, value in criteria.items()
    if not (value is True or value == 'PASS')
}
assert blocking_criteria, 'Stage 4 was required without a recorded failed criterion'

skip = {
    'notebook': '07',
    'status': 'SKIPPED_PREREQUISITE',
    'science_executed': False,
    'model_inference_run': False,
    'scope': 'cross-scale P1',
    'blocking_criteria': blocking_criteria,
    'next': '08_report_stage4',
}
assert skip['status'] == 'SKIPPED_PREREQUISITE'
assert skip['science_executed'] is False
assert skip['model_inference_run'] is False
repair.setdefault('stage3_notebooks', {})['07'] = skip
(ROOT / 'results/metrics.json').write_text(
    json.dumps(metrics, indent=2, sort_keys=True) + chr(10)
)
print(json.dumps(skip, indent=2))

{
  "notebook": "07",
  "status": "SKIPPED_PREREQUISITE",
  "science_executed": false,
  "model_inference_run": false,
  "scope": "cross-scale P1",
  "blocking_criteria": {
    "capability_preserved": false,
    "g_pos_reproduced": false
  },
  "next": "08_report_stage4"
}
